# Small RAG System
**Topic:** RAG (retrieval + generation), chunking, embeddings
**Instructions:** Upload a PDF (4-15 pages, containing real facts like numbers/dates), set your GEMINI_API_KEY in Colab Secrets, and run all cells.

In [ ]:
!pip install -q pypdf google-genai numpy
import pypdf
import numpy as np
from google import genai
from google.colab import userdata

# Initialize the client
client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))

### Step 1: Upload PDF and read the text

In [ ]:
# REPLACE with the name of the PDF you uploaded to Colab
pdf_filename = 'your_document.pdf'

text = ''
with open(pdf_filename, 'rb') as f:
    reader = pypdf.PdfReader(f)
    for page in reader.pages:
        # Extract text from each page
        extracted = page.extract_text()
        if extracted:
            text += extracted + '\n'

print(f'Total characters extracted: {len(text)}')


### Step 2: Split text into chunks (800 chars, 150 overlap)

In [ ]:
chunk_size = 800
overlap = 150
chunks = []
start = 0

while start < len(text):
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start += (chunk_size - overlap)

print(f'Total chunks: {len(chunks)}\n')
print('--- Sample Chunk ---')
print(chunks[0])


### Step 3: Convert every chunk into an embedding

In [ ]:
chunk_data = []

# We use text-embedding-004 to create embeddings
for i, chunk in enumerate(chunks):
    # Process one by one (or you could batch them if preferred)
    response = client.models.embed_content(
        model='text-embedding-004',
        contents=chunk
    )
    chunk_data.append({
        'text': chunk,
        'embedding': response.embeddings[0].values
    })

print(f'Created embeddings for {len(chunk_data)} chunks.')


### Step 4: Write the search step

In [ ]:
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def search(query, data, top_k=3):
    # Embed the question
    response = client.models.embed_content(
        model='text-embedding-004',
        contents=query
    )
    query_embedding = response.embeddings[0].values
    
    # Compare against all chunk embeddings
    results = []
    for item in data:
        sim = cosine_similarity(query_embedding, item['embedding'])
        results.append({
            'text': item['text'],
            'score': sim
        })
        
    # Sort by highest similarity first and pick top 3
    results.sort(key=lambda x: x['score'], reverse=True)
    return results[:top_k]


### Step 5: Test the search step alone

In [ ]:
# REPLACE this with a question whose answer is in your PDF
test_question = 'What are the fees or timings mentioned?'

top_chunks = search(test_question, chunk_data, top_k=3)

print(f'Search results for: "{test_question}"\n')
for i, chunk_info in enumerate(top_chunks):
    print(f'--- Top {i+1} Chunk (Score: {chunk_info["score"]:.4f}) ---')
    print(chunk_info['text'])
    print()


### Step 6 & 7: Final answering step with grounding

In [ ]:
def ask_bot(query, data):
    top_chunks = search(query, data, top_k=3)
    
    # Combine the top 3 chunks into a single context string
    context = '\n\n---\n\n'.join([c['text'] for c in top_chunks])
    
    prompt = f"""You are a helpful assistant.
Answer the user's question using ONLY the information in the provided document text below. 
If the answer is not contained in the text, you must say "I don't know" or "The document does not contain this information". Do not guess.

Document Text:
{context}

Question: {query}
"""
    
    # Using gemini-2.5-flash for the generation step
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=prompt
    )
    
    print('Question:', query)
    print('Answer:', response.text)
    print('\nSources used:')
    for i, c in enumerate(top_chunks):
         # Print a snippet of the chunk and its score
         snippet = c['text'][:100].replace('\n', ' ')
         print(f'[Chunk {i+1} | Score: {c["score"]:.4f}] {snippet}...')
    print('-' * 50)


### Tests: 6 Questions (3 In-Document, 3 Out-of-Document)

In [ ]:
# REPLACE these with questions relevant to your PDF.
# Make sure at least one in-document question asks for a number, date, or amount.
in_doc_questions = [
    'What is the total fee or specific date mentioned?', 
    'What are the main rules in the first section?',
    'Who is the target audience or what are the steps to follow?'
]

# Questions completely unrelated to your PDF
out_doc_questions = [
    'What is the capital of France?',
    'How do I cook pasta?',
    'What is the plot of the movie Inception?'
]

print('=== IN-DOCUMENT QUESTIONS ===\n')
for q in in_doc_questions:
    ask_bot(q, chunk_data)
    
print('\n=== OUT-OF-DOCUMENT QUESTIONS ===\n')
for q in out_doc_questions:
    ask_bot(q, chunk_data)


#### Conclusion on retrieved answers:
*(Replace with your own words after running)*
The answers to the in-document questions were correct and directly pulled from the retrieved chunks, confirming the retrieval worked. For the out-of-document questions, the bot correctly admitted it didn't know, proving the prompt constraints successfully prevented hallucinations.

### Experiment: Chunk Size 3000

In [ ]:
exp_chunk_size = 3000
exp_overlap = 150
exp_chunks = []
start = 0

while start < len(text):
    end = start + exp_chunk_size
    chunk = text[start:end]
    exp_chunks.append(chunk)
    start += (exp_chunk_size - exp_overlap)

print(f'Total chunks (size 3000): {len(exp_chunks)}\n')

exp_chunk_data = []
for chunk in exp_chunks:
    response = client.models.embed_content(
        model='text-embedding-004',
        contents=chunk
    )
    exp_chunk_data.append({
        'text': chunk,
        'embedding': response.embeddings[0].values
    })

print('Running in-document questions with larger chunks:\n')
for q in in_doc_questions:
    ask_bot(q, exp_chunk_data)


#### Conclusion on Chunk Size Experiment:
*(Replace with your own words after running)*
When the chunk size was increased to 3000, the total number of chunks decreased significantly. The similarity scores changed because each chunk now contains a broader mix of topics, diluting specific keywords. While the model still answered correctly due to a larger context window, retrieval precision for highly specific facts slightly declined compared to the 800-character chunks.